# SurvivalData playground

An editable companion to the [SurvivalData guide](survival-data.md). All data
here are synthetic: six patients, time in months since diagnosis, and disease
progression as the event. survjax can currently build and check observations;
likelihoods are not implemented yet.

Change any value and rerun the cell. The last section lists things to try.

In [ ]:
import jax
import jax.numpy as jnp

import survjax

nan, inf = jnp.nan, jnp.inf
print(f"JAX float64 enabled: {jax.config.jax_enable_x64}")
data = survjax.SurvivalData(jnp.array([4.0, 12.0, 7.0, 9.0]), event=[1, 0, 1, 0])

## Time and status

Patients 1 and 3 progressed at months 4 and 7 (`event=1`). Patients 2 and 4
were last seen without progression at months 12 and 9 (`event=0`), so they are
right censored. The printout shows an exact time for events and the set of
possible event times, `(lower, upper]`, for censored rows.

In [ ]:
data

The printout is built from five stored arrays with one entry per patient.
`exact` marks events, `lower` and `upper` bound the event time, and the
truncation window `(left_trunc, right_trunc]` defaults to `(0, inf]`.

In [ ]:
print("exact:      ", data.exact)
print("lower:      ", data.lower)
print("upper:      ", data.upper)
print("left_trunc: ", data.left_trunc)
print("right_trunc:", data.right_trunc)

## The full cohort

Two more patients joined two months after diagnosis, so both are left
truncated at month 2. Patient 5 had a clear scan at month 6 and progression at
month 9 (interval censored). Patient 6 already showed progression at the first
scan at month 3 (left censored). Keyword flags describe each kind; a row
with no flag is an exact event, and `time2` is read only on interval rows.

In [ ]:
cohort = survjax.SurvivalData(
    [4.0, 12.0, 7.0, 9.0, 6.0, 3.0],
    right_censored=[False, True, False, True, False, False],
    left_censored=[False, False, False, False, False, True],
    interval_censored=[False, False, False, False, True, False],
    time2=[nan, nan, nan, nan, 9.0, nan],
    left_trunc=[0.0, 0.0, 0.0, 0.0, 2.0, 2.0],
)
cohort

## The same cohort as endpoints

`from_bounds` takes each patient's set directly. Exact events need the
`exact` mask; their upper endpoint may be `nan`. The printout should match the
one above.

In [ ]:
same_cohort = survjax.SurvivalData.from_bounds(
    [4.0, 12.0, 7.0, 9.0, 6.0, 0.0],
    [nan, inf, nan, inf, 9.0, 3.0],
    exact=[True, False, True, False, False, False],
    left_trunc=[0.0, 0.0, 0.0, 0.0, 2.0, 2.0],
)
same_cohort

## Checking data

Construction never stops on bad values. It returns the data, notes problems
in a footer, and records them in `data.issues`. These five records each have
a problem: an exact event at month 0, an interval with no end, an interval
that ends before it starts, a patient censored before entering the study, and
an interval `(0, inf]` that rules nothing out (valid, but uninformative).

In [ ]:
bad = survjax.SurvivalData(
    [0.0, 6.0, 9.0, 1.0, 0.0],
    right_censored=[False, False, False, True, False],
    interval_censored=[False, True, True, False, True],
    time2=[nan, nan, 6.0, nan, inf],
    left_trunc=[0.0, 0.0, 0.0, 2.0, 0.0],
)
bad

In [ ]:
bad.issues

Each rule is also a boolean mask with one entry per row, and `ok` is one
boolean for the whole data set:

In [ ]:
print("invalid_rows: ", bad.issues.invalid_rows)
print("invalid_time: ", bad.issues.invalid_time)
print("uninformative:", bad.issues.uninformative)
print("ok:           ", bad.issues.ok)

To stop a script on bad data, call `raise_if_invalid()`. Here the error is
caught and printed so that the notebook keeps running:

In [ ]:
try:
    bad.issues.raise_if_invalid()
except survjax.InvalidDataError as error:
    print(error)

## Inside `jax.jit`

Construction and `data.issues` work in compiled code. Return the issues and
check them afterwards. Printing inside the compiled function shows only the
shape, because the values are not known there yet.

In [ ]:
@jax.jit
def diagnose(time, status):
    data = survjax.SurvivalData(time, status)
    print("traced:", data)
    return data.issues


diagnose(jnp.array([4.0, 12.0, 7.0]), jnp.array([1, 0, 2]))

## Things to try

- Give patient 1 status `2`, as in R's 1/2 coding. Which rule fires?
- In `from_bounds`, set patient 1's upper endpoint to `4.0` and drop
  `exact=True` for that row. Equal endpoints alone are not an event.
- Move patient 5's entry (`left_trunc`) to month 10, after its interval.
- Add `right_trunc=24.0` to the cohort and look at the window column.
- Restart JupyterLab with `JAX_ENABLE_X64=1` in the environment to see float64.